# MLflow GenAI Evaluation – Results Visualisation

This notebook runs the evaluation subcommands from `scripts/mlflow/vanilla.py`
and visualises the results inline.

## Prerequisites

1. Start the MLflow tracking server in a separate terminal:
   ```bash
   make mlflow
   ```
2. The `mlflow>=3.12.0` dependency is already declared in `pyproject.toml`.

## References

- [MLflow GenAI Evaluation docs](https://mlflow.org/docs/latest/genai/)
- [Tutorial Step 5](../docs/tutorial/05-mlflow-genai-evaluation.md)


In [ ]:
# Cell 1 – verify CLI help
!uv run python ../../scripts/mlflow/vanilla.py --help

## 1. Configure MLflow

Point the client at the local tracking server started by `make mlflow`.
Adjust `MLFLOW_TRACKING_URI` if you are using a different port or host.

In [ ]:
import os
import sys

# Make the repository root importable so concierge.settings is available.
REPO_ROOT = os.path.abspath("../..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import mlflow  # noqa: E402

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "microsoft-foundry-vanilla")

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Tracking URI : {TRACKING_URI}")
print(f"Experiment   : {EXPERIMENT_NAME}")
print(f"MLflow version: {mlflow.__version__}")

## 2. Inline evaluation dataset

Define the same five QA pairs used by `scripts/mlflow/vanilla.py`.

In [ ]:
import pandas as pd

DATASET = [
    {"inputs": {"question": "What is the capital of France?"}, "expected_response": "Paris"},
    {"inputs": {"question": "What is 2 + 2?"}, "expected_response": "4"},
    {"inputs": {"question": "Who wrote Romeo and Juliet?"}, "expected_response": "William Shakespeare"},
    {"inputs": {"question": "What is the boiling point of water in Celsius?"}, "expected_response": "100"},
    {"inputs": {"question": "What planet is closest to the Sun?"}, "expected_response": "Mercury"},
]

df_dataset = pd.DataFrame(
    [{"question": row["inputs"]["question"], "expected_response": row["expected_response"]} for row in DATASET]
)
df_dataset

## 3. Target application: simple QA function

The same deterministic QA function used in `scripts/mlflow/vanilla.py`.

In [ ]:
def simple_qa(question: str) -> str:
    """Return hard-coded answers for the built-in dataset."""
    answers = {
        "what is the capital of france?": "Paris",
        "what is 2 + 2?": "4",
        "who wrote romeo and juliet?": "William Shakespeare",
        "what is the boiling point of water in celsius?": "100 degrees Celsius",
        "what planet is closest to the sun?": "Mercury",
    }
    return answers.get(question.lower().strip(), "I don't know.")


# Quick smoke test
for row in DATASET:
    q = row["inputs"]["question"]
    print(f"Q: {q}")
    print(f"A: {simple_qa(q)}\n")

## 4. Heuristic evaluation (no Azure required)

Define three pure-Python scorers and run `mlflow.genai.evaluate()`.

In [ ]:
from mlflow.genai.scorers import scorer


@scorer
def exact_match(outputs: str, expected_response: str) -> float:
    """1.0 when output == expected_response (case-insensitive)."""
    return 1.0 if outputs.strip().lower() == expected_response.strip().lower() else 0.0


@scorer
def contains(outputs: str, expected_response: str) -> float:
    """1.0 when output contains expected_response."""
    return 1.0 if expected_response.strip().lower() in outputs.strip().lower() else 0.0


@scorer
def non_empty(outputs: str) -> float:
    """1.0 when output is non-empty."""
    return 1.0 if outputs.strip() else 0.0


heuristic_results = mlflow.genai.evaluate(
    data=DATASET,
    predict_fn=lambda inputs: simple_qa(inputs["question"]),
    scorers=[exact_match, contains, non_empty],
    run_name="notebook-heuristic-eval",
)

print(heuristic_results.metrics_summary())

### 4.1 Per-row heuristic results

In [ ]:
df_heuristic = heuristic_results.tables["eval_results"]
df_heuristic

### 4.2 Visualise heuristic scores

In [ ]:
import matplotlib.pyplot as plt

score_cols = [c for c in df_heuristic.columns if c in ("exact_match", "contains", "non_empty")]
if score_cols and "inputs/question" in df_heuristic.columns:
    ax = df_heuristic.set_index("inputs/question")[score_cols].plot(
        kind="bar",
        figsize=(10, 4),
        title="Heuristic Scorer Results",
        ylim=(0, 1.1),
        rot=30,
    )
    ax.set_ylabel("Score")
    plt.tight_layout()
    plt.show()
else:
    print("Score columns not found – check column names above.")
    print(df_heuristic.columns.tolist())

## 5. Custom scorer: token overlap

In [ ]:
@scorer
def token_overlap(outputs: str, expected_response: str) -> float:
    """Jaccard similarity between output and expected_response token sets."""
    a = set(outputs.lower().split())
    b = set(expected_response.lower().split())
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


custom_results = mlflow.genai.evaluate(
    data=DATASET,
    predict_fn=lambda inputs: simple_qa(inputs["question"]),
    scorers=[token_overlap],
    run_name="notebook-custom-scorer-eval",
)

print(custom_results.metrics_summary())

### 5.1 Per-row custom scorer results

In [ ]:
df_custom = custom_results.tables["eval_results"]
df_custom

## 6. Compare evaluation runs

Search all runs in the current experiment and display a side-by-side metrics table.

In [ ]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    max_results=20,
    output_format="pandas",
)

keep_cols = [
    c for c in runs.columns if c.startswith("metrics.") or c in {"run_id", "tags.mlflow.runName", "start_time"}
]
df_compare = runs[keep_cols].rename(columns={"tags.mlflow.runName": "run_name"})
df_compare = df_compare.sort_values("start_time", ascending=False) if "start_time" in df_compare.columns else df_compare
df_compare

### 6.1 Aggregate metrics bar chart

In [ ]:
metric_cols = [c for c in df_compare.columns if c.startswith("metrics.")]
if metric_cols and "run_name" in df_compare.columns:
    plot_df = df_compare.set_index("run_name")[metric_cols]
    plot_df.columns = [c.replace("metrics.", "") for c in plot_df.columns]
    ax = plot_df.plot(
        kind="bar",
        figsize=(12, 5),
        title="Evaluation Run Comparison",
        ylim=(0, 1.1),
        rot=20,
    )
    ax.set_ylabel("Score")
    plt.tight_layout()
    plt.show()
else:
    print("No metric columns found or no run_name column. Run evaluate / custom-scorer first.")

## 7. Open MLflow UI

All runs are also visible in the interactive MLflow UI:

```
http://127.0.0.1:5000
```

Navigate to **Experiments → [experiment name] → Evaluation** for
colour-coded comparisons, trace drill-downs, and run diffs.